# Hybrid Quantum Biosignature Advantage Notebook

This notebook rebuilds the original `hybrid_quantum_biosignature.ipynb` workflow around a **true 12-qubit quantum bottleneck** and an experiment suite that asks a harder question: does the quantum block actually earn its cost relative to matched classical alternatives?

## Design Goals

- Keep the Ariel ADC2023 loading and scaling pipeline from the original notebook.
- Constrain the main hybrid path to `<= 12` qubits.
- Make the quantum block harder to bypass than in the original architecture.
- Add fair baselines, low-data experiments, and an OOD split.
- Include a preflight timing check before committing to long runs.

## Recommended Usage

1. Run the setup, data, and model-definition sections first.
2. Start in `dev` mode to validate runtime and debug the circuit path.
3. Run the preflight benchmark cell before the full suite.
4. Only then switch to `full` mode for the heavier experiment matrix.

In [1]:
import json
import math
import os
import random
import time
from copy import deepcopy
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

os.environ.setdefault("XDG_CACHE_HOME", str(Path(".cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.model_selection import train_test_split

try:
    from tqdm.auto import tqdm
except ImportError:
    class _TqdmFallback:
        def __init__(self, iterable=None, total=None, desc=None, leave=True):
            self.iterable = iterable

        def __iter__(self):
            return iter(self.iterable)

        def set_postfix(self, **kwargs):
            pass

        @staticmethod
        def write(message):
            print(message)

    def tqdm(iterable=None, total=None, desc=None, leave=True):
        return _TqdmFallback(iterable=iterable, total=total, desc=desc, leave=leave)


AUX_FEATURE_COLS = [
    "star_mass_kg",
    "star_radius_m",
    "star_temperature",
    "planet_mass_kg",
    "planet_orbital_period",
    "planet_distance",
    "planet_surface_gravity",
    "log10_noise_mean",
]
TARGET_COLS = ["log_H2O", "log_CO2", "log_CO", "log_CH4", "log_NH3"]
QUANTUM_FEATURE_NAMES = [
    "band_1p0",
    "band_1p4",
    "band_1p9",
    "band_2p7",
    "band_4p3",
    "band_6p3",
    "blue_red_slope",
    "mid_curvature",
    "star_temperature",
    "planet_surface_gravity",
    "planet_distance",
    "log10_noise_mean",
]

SEED = 42
DATA_ROOT_CANDIDATES = [
    Path("FullDataset"),
    Path("ariel-ml-dataset"),
    Path("../FullDataset"),
    Path("../ariel-ml-dataset"),
]
EXPERIMENT_MODE = os.environ.get("HQB_MODE", "dev").strip().lower()
if EXPERIMENT_MODE not in {"dev", "full"}:
    raise ValueError(f"Unsupported EXPERIMENT_MODE={EXPERIMENT_MODE!r}")

MODE_CFG = {
    "dev": {
        "output_dir": Path("outputs/hybrid_quantum_biosignature_advantage/dev"),
        "train_batch_size": 48,
        "eval_batch_size": 512,
        "max_epochs": 8,
        "warmup_epochs": 2,
        "ramp_epochs": 2,
        "early_stop": 3,
        "scheduler_patience": 2,
        "train_subset_cap": 2048,
        "val_subset_cap": 1024,
        "preflight_batches": 1,
    },
    "full": {
        "output_dir": Path("outputs/hybrid_quantum_biosignature_advantage/full"),
        "train_batch_size": 64,
        "eval_batch_size": 2048,
        "max_epochs": 20,
        "warmup_epochs": 3,
        "ramp_epochs": 3,
        "early_stop": 5,
        "scheduler_patience": 3,
        "train_subset_cap": None,
        "val_subset_cap": None,
        "preflight_batches": 2,
    },
}

CONFIG = {
    **MODE_CFG[EXPERIMENT_MODE],
    "val_fraction": 0.10,
    "test_fraction": 0.10,
    "classical_lr": 1.5e-3,
    "quantum_lr": 4e-4,
    "weight_decay": 1e-4,
    "gradient_clip": 5.0,
    "quantum_gradient_clip": 1.0,
    "dropout": 0.08,
    "num_qubits": 12,
    "num_qblocks": 2,
    "readout_pairs": [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11)],
    "aux_hidden": 32,
    "aux_out": 8,
    "spec_hidden": 24,
    "spec_out": 8,
    "interface_hidden": 32,
    "skip_dim": 8,
    "head_hidden": 48,
    "ood_target": "log_H2O",
    "ood_quantile": 0.85,
    "low_data_sizes": [500, 1000, 2000, 5000],
    "run_experiments": False,
    "save_artifacts": True,
}
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)


def set_random_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_data_root(candidates: Iterable[Path]) -> Path:
    for root in candidates:
        train_dir = root / "TrainingData"
        if (train_dir / "AuxillaryTable.csv").exists() and (train_dir / "SpectralData.hdf5").exists():
            return root
    searched = "\n".join(f" - {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        "Could not locate the Ariel dataset. Put the extracted dataset in one of:\n"
        f"{searched}"
    )


def resolve_quantum_device() -> str:
    for name in ("lightning.qubit", "default.qubit"):
        try:
            kwargs = {"wires": 1}
            if name.startswith("lightning."):
                kwargs["c_dtype"] = np.complex64
            qml.device(name, **kwargs)
            return name
        except Exception:
            continue
    raise RuntimeError("No supported PennyLane device found. Install pennylane-lightning or use default.qubit.")


set_random_seed(SEED)
DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
QUANTUM_DEVICE = resolve_quantum_device()

print(json.dumps(CONFIG, indent=2, default=str))
print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Torch device: {DEVICE} | Quantum device: {QUANTUM_DEVICE}")

{
  "output_dir": "outputs/hybrid_quantum_biosignature_advantage/dev",
  "train_batch_size": 48,
  "eval_batch_size": 512,
  "max_epochs": 8,
  "warmup_epochs": 2,
  "ramp_epochs": 2,
  "early_stop": 3,
  "scheduler_patience": 2,
  "train_subset_cap": 2048,
  "val_subset_cap": 1024,
  "preflight_batches": 1,
  "val_fraction": 0.1,
  "test_fraction": 0.1,
  "classical_lr": 0.0015,
  "quantum_lr": 0.0004,
  "weight_decay": 0.0001,
  "gradient_clip": 5.0,
  "quantum_gradient_clip": 1.0,
  "dropout": 0.08,
  "num_qubits": 12,
  "num_qblocks": 2,
  "readout_pairs": [
    [
      0,
      1
    ],
    [
      2,
      3
    ],
    [
      4,
      5
    ],
    [
      6,
      7
    ],
    [
      8,
      9
    ],
    [
      10,
      11
    ]
  ],
  "aux_hidden": 32,
  "aux_out": 8,
  "spec_hidden": 24,
  "spec_out": 8,
  "interface_hidden": 32,
  "skip_dim": 8,
  "head_hidden": 48,
  "ood_target": "log_H2O",
  "ood_quantile": 0.85,
  "low_data_sizes": [
    500,
    1000,
    2000,
    5

/Users/jkw/Documents/uni/axion/hack4sages/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Loading And Feature Engineering

This section keeps the original Ariel data-loading contract, then adds a compact 12-feature bank for the quantum bottleneck. The feature bank mixes molecule-sensitive spectral summaries, a few high-value auxiliary features, and a noise descriptor so the quantum circuit models cross-feature interactions instead of arbitrary compression.

In [2]:
train_dir = DATA_ROOT / "TrainingData"
aux_path = train_dir / "AuxillaryTable.csv"
gt_path = train_dir / "Ground Truth Package" / "FM_Parameter_Table.csv"
spectral_path = train_dir / "SpectralData.hdf5"

aux_df = pd.read_csv(aux_path)
gt_df = pd.read_csv(gt_path)
labels = aux_df.merge(gt_df[["planet_ID", *TARGET_COLS]], on="planet_ID", how="inner").reset_index(drop=True)

with h5py.File(spectral_path, "r") as handle:
    first_key = list(handle.keys())[0]
    wavelength_um = np.asarray(handle[first_key]["instrument_wlgrid"][:], dtype=np.float32)
    noisy_spectra = np.empty((len(labels), len(wavelength_um)), dtype=np.float32)
    noise_arr = np.empty_like(noisy_spectra)
    for i, pid in enumerate(labels["planet_ID"].values):
        grp = handle[f"Planet_{pid}"]
        noisy_spectra[i] = grp["instrument_spectrum"][:]
        noise_arr[i] = grp["instrument_noise"][:]

labels["log10_noise_mean"] = np.log10(np.clip(noise_arr.mean(axis=1), 1e-10, None))
aux_raw = labels[AUX_FEATURE_COLS].to_numpy(dtype=np.float32)
targets_raw = labels[TARGET_COLS].to_numpy(dtype=np.float32)

per_sample_mean = noisy_spectra.mean(axis=1, keepdims=True)
per_sample_mean = np.where(per_sample_mean == 0, 1.0, per_sample_mean)
spectra_norm = (noisy_spectra / per_sample_mean).astype(np.float32)
spectra_raw = spectra_norm[:, None, :]


def fit_scaler(arr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = arr.astype(np.float64).mean(axis=0).astype(np.float32)
    scale = arr.astype(np.float64).std(axis=0).astype(np.float32)
    scale = np.where(scale == 0, 1.0, scale)
    return mean, scale


def band_mean(spectra_2d: np.ndarray, wavelengths: np.ndarray, center: float, half_width: float) -> np.ndarray:
    mask = np.abs(wavelengths - center) <= half_width
    if not np.any(mask):
        idx = int(np.argmin(np.abs(wavelengths - center)))
        return spectra_2d[:, idx]
    return spectra_2d[:, mask].mean(axis=1)


def extract_quantum_feature_bank(
    spectra_2d: np.ndarray,
    wavelengths: np.ndarray,
    aux_table: pd.DataFrame,
    noise_matrix: np.ndarray,
) -> np.ndarray:
    bands = [
        band_mean(spectra_2d, wavelengths, center=1.0, half_width=0.12),
        band_mean(spectra_2d, wavelengths, center=1.4, half_width=0.12),
        band_mean(spectra_2d, wavelengths, center=1.9, half_width=0.16),
        band_mean(spectra_2d, wavelengths, center=2.7, half_width=0.18),
        band_mean(spectra_2d, wavelengths, center=4.3, half_width=0.22),
        band_mean(spectra_2d, wavelengths, center=6.3, half_width=0.28),
    ]
    left = band_mean(spectra_2d, wavelengths, center=0.8, half_width=0.18)
    right = band_mean(spectra_2d, wavelengths, center=6.8, half_width=0.25)
    midpoint = band_mean(spectra_2d, wavelengths, center=3.2, half_width=0.20)
    slope = right - left
    curvature = midpoint - 0.5 * (left + right)
    log_noise = np.log10(np.clip(noise_matrix.mean(axis=1), 1e-10, None)).astype(np.float32)

    feature_bank = np.stack(
        [
            *bands,
            slope.astype(np.float32),
            curvature.astype(np.float32),
            aux_table["star_temperature"].to_numpy(dtype=np.float32),
            aux_table["planet_surface_gravity"].to_numpy(dtype=np.float32),
            aux_table["planet_distance"].to_numpy(dtype=np.float32),
            log_noise,
        ],
        axis=1,
    )
    return feature_bank.astype(np.float32)


feature_cache_path = CONFIG["output_dir"] / "quantum_feature_bank_cache.npz"
if feature_cache_path.exists():
    cache = np.load(feature_cache_path)
    quantum_feature_raw = cache["quantum_feature_raw"].astype(np.float32)
else:
    quantum_feature_raw = extract_quantum_feature_bank(spectra_norm, wavelength_um, labels, noise_arr)
    np.savez_compressed(feature_cache_path, quantum_feature_raw=quantum_feature_raw)

print(f"Samples: {len(labels):,}")
print(f"Spectra shape: {spectra_raw.shape} | Quantum feature bank shape: {quantum_feature_raw.shape}")
print(f"Wavelength range: {wavelength_um.min():.2f} - {wavelength_um.max():.2f} um")
print("Quantum feature names:", QUANTUM_FEATURE_NAMES)

Samples: 41,423
Spectra shape: (41423, 1, 52) | Quantum feature bank shape: (41423, 12)
Wavelength range: 0.55 - 7.28 um
Quantum feature names: ['band_1p0', 'band_1p4', 'band_1p9', 'band_2p7', 'band_4p3', 'band_6p3', 'blue_red_slope', 'mid_curvature', 'star_temperature', 'planet_surface_gravity', 'planet_distance', 'log10_noise_mean']


In [3]:
def subset_indices(indices: np.ndarray, limit: Optional[int], seed: int) -> np.ndarray:
    if limit is None or len(indices) <= limit:
        return np.sort(indices)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(indices, size=limit, replace=False))


def build_iid_split(n_samples: int, seed: int, test_fraction: float, val_fraction: float):
    all_idx = np.arange(n_samples)
    train_val_idx, test_idx = train_test_split(
        all_idx, test_size=test_fraction, random_state=seed, shuffle=True
    )
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=val_fraction, random_state=seed, shuffle=True
    )
    return np.sort(train_idx), np.sort(val_idx), np.sort(test_idx)


def build_quantile_ood_split(
    targets: np.ndarray,
    target_cols: list[str],
    target_name: str,
    quantile: float,
    seed: int,
    val_fraction: float,
):
    target_idx = target_cols.index(target_name)
    threshold = np.quantile(targets[:, target_idx], quantile)
    ood_mask = targets[:, target_idx] >= threshold
    ood_test_idx = np.flatnonzero(ood_mask)
    in_domain_idx = np.flatnonzero(~ood_mask)
    train_idx, val_idx = train_test_split(
        in_domain_idx, test_size=val_fraction, random_state=seed, shuffle=True
    )
    return np.sort(train_idx), np.sort(val_idx), np.sort(ood_test_idx), float(threshold)


iid_train_idx, iid_val_idx, iid_test_idx = build_iid_split(
    len(labels),
    seed=SEED,
    test_fraction=CONFIG["test_fraction"],
    val_fraction=CONFIG["val_fraction"],
)
ood_train_idx, ood_val_idx, ood_test_idx, ood_threshold = build_quantile_ood_split(
    targets_raw,
    TARGET_COLS,
    target_name=CONFIG["ood_target"],
    quantile=CONFIG["ood_quantile"],
    seed=SEED,
    val_fraction=CONFIG["val_fraction"],
)

aux_mean, aux_scale = fit_scaler(aux_raw[iid_train_idx])
spec_mean, spec_scale = fit_scaler(spectra_raw[iid_train_idx, 0, :])
target_mean, target_scale = fit_scaler(targets_raw[iid_train_idx])
qfeat_mean, qfeat_scale = fit_scaler(quantum_feature_raw[iid_train_idx])


def scale_aux(a: np.ndarray) -> np.ndarray:
    return ((a - aux_mean) / aux_scale).astype(np.float32)


def scale_spec(s: np.ndarray) -> np.ndarray:
    return ((s - spec_mean[None, None, :]) / spec_scale[None, None, :]).astype(np.float32)


def scale_target(t: np.ndarray) -> np.ndarray:
    return ((t - target_mean) / target_scale).astype(np.float32)


def scale_qfeat(q: np.ndarray) -> np.ndarray:
    return ((q - qfeat_mean) / qfeat_scale).astype(np.float32)


def unscale_target(t: np.ndarray) -> np.ndarray:
    return (t * target_scale + target_mean).astype(np.float32)


def make_split_tensors(indices: np.ndarray) -> dict[str, Any]:
    return {
        "indices": np.asarray(indices, dtype=np.int64),
        "planet_ids": labels.iloc[indices]["planet_ID"].to_numpy(),
        "aux": torch.from_numpy(scale_aux(aux_raw[indices])),
        "spectra": torch.from_numpy(scale_spec(spectra_raw[indices])),
        "qfeat": torch.from_numpy(scale_qfeat(quantum_feature_raw[indices])),
        "target": torch.from_numpy(scale_target(targets_raw[indices])),
        "target_raw": targets_raw[indices].copy(),
    }


def build_bundle(name: str, train_idx: np.ndarray, val_idx: np.ndarray, test_idx: np.ndarray) -> dict[str, Any]:
    return {
        "name": name,
        "train": make_split_tensors(train_idx),
        "val": make_split_tensors(val_idx),
        "test": make_split_tensors(test_idx),
    }


iid_bundle = build_bundle("iid", iid_train_idx, iid_val_idx, iid_test_idx)
ood_bundle = build_bundle("ood_high_h2o", ood_train_idx, ood_val_idx, ood_test_idx)

print(f"IID split sizes: train={len(iid_train_idx):,}, val={len(iid_val_idx):,}, test={len(iid_test_idx):,}")
print(f"OOD split sizes: train={len(ood_train_idx):,}, val={len(ood_val_idx):,}, test={len(ood_test_idx):,}")
print(f"OOD threshold ({CONFIG['ood_target']}): {ood_threshold:.3f}")

IID split sizes: train=33,552, val=3,728, test=4,143
OOD split sizes: train=31,688, val=3,521, test=6,214
OOD threshold (log_H2O): -3.888


## Model Definitions

The new default model is a **12-qubit bottleneck regressor**. A small classical stem produces bounded 12-angle inputs and a tiny classical skip, but the main predictive path goes through the quantum readout. Baselines reuse the same input contract so the comparison stays fair.

In [4]:
class AuxStem(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SpectralStem(nn.Module):
    def __init__(self, hidden_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(16, hidden_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(self.conv(x))


class QuantumInterface(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, angle_dim: int, skip_dim: int, dropout: float):
        super().__init__()
        self.feature_mixer = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.angle_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, angle_dim),
            nn.LayerNorm(angle_dim),
        )
        self.skip_proj = nn.Sequential(
            nn.Linear(hidden_dim, skip_dim),
            nn.GELU(),
            nn.Linear(skip_dim, skip_dim),
        )

    def forward(self, aux_feat: torch.Tensor, spec_feat: torch.Tensor, qfeat: torch.Tensor):
        hidden = self.feature_mixer(torch.cat([aux_feat, spec_feat, qfeat], dim=-1))
        angles = torch.tanh(self.angle_proj(hidden)) * math.pi
        skip = self.skip_proj(hidden)
        return angles, skip, hidden


class ReuploadQuantumBlock(nn.Module):
    def __init__(self, num_qubits: int, num_blocks: int, readout_pairs: list[tuple[int, int]], quantum_device_name: str):
        super().__init__()
        self.num_qubits = num_qubits
        self.num_blocks = num_blocks
        self.readout_pairs = list(readout_pairs)
        self.input_scale = nn.Parameter(torch.ones(num_blocks, num_qubits, dtype=torch.float32))
        self.input_bias = nn.Parameter(torch.zeros(num_blocks, num_qubits, dtype=torch.float32))
        self.weights = nn.Parameter(0.05 * torch.randn(num_blocks, num_qubits, 3, dtype=torch.float32))
        self.output_dim = num_qubits + len(self.readout_pairs)

        dev_kwargs = {"wires": num_qubits}
        if quantum_device_name.startswith("lightning."):
            dev_kwargs["c_dtype"] = np.complex64
        dev = qml.device(quantum_device_name, **dev_kwargs)

        @qml.qnode(dev, interface="torch", diff_method="adjoint")
        def circuit(inputs, input_scale, input_bias, weights):
            for block in range(num_blocks):
                for q in range(num_qubits):
                    angle = input_scale[block, q] * inputs[..., q] + input_bias[block, q]
                    qml.RY(angle, wires=q)
                    qml.RZ(0.5 * angle, wires=q)
                for q in range(num_qubits):
                    qml.RY(weights[block, q, 0], wires=q)
                    qml.RZ(weights[block, q, 1], wires=q)
                for q in range(num_qubits):
                    qml.CZ(wires=[q, (q + 1) % num_qubits])
                for q in range(0, num_qubits, 2):
                    qml.CRX(weights[block, q, 2], wires=[q, (q + 2) % num_qubits])
            measurements = [qml.expval(qml.PauliZ(q)) for q in range(num_qubits)]
            for left, right in readout_pairs:
                measurements.append(qml.expval(qml.PauliZ(left) @ qml.PauliZ(right)))
            return measurements

        self.qnode = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.qnode(x.float(), self.input_scale, self.input_bias, self.weights)
        if isinstance(out, (list, tuple)):
            out = torch.stack(tuple(out), dim=-1)
        return out.float().to(x.device)


class BottleneckRegressor(nn.Module):
    def __init__(self, backend: str = "quantum"):
        super().__init__()
        if backend not in {"quantum", "classical", "frozen_quantum", "identity"}:
            raise ValueError(f"Unsupported backend: {backend}")
        self.backend = backend
        self.aux_stem = AuxStem(len(AUX_FEATURE_COLS), CONFIG["aux_hidden"], CONFIG["aux_out"], CONFIG["dropout"])
        self.spec_stem = SpectralStem(CONFIG["spec_hidden"], CONFIG["spec_out"], CONFIG["dropout"])
        interface_in = CONFIG["aux_out"] + CONFIG["spec_out"] + len(QUANTUM_FEATURE_NAMES)
        self.interface = QuantumInterface(
            interface_in,
            CONFIG["interface_hidden"],
            CONFIG["num_qubits"],
            CONFIG["skip_dim"],
            CONFIG["dropout"],
        )
        if backend in {"quantum", "frozen_quantum"}:
            self.bottleneck = ReuploadQuantumBlock(
                CONFIG["num_qubits"],
                CONFIG["num_qblocks"],
                CONFIG["readout_pairs"],
                QUANTUM_DEVICE,
            )
            bottleneck_out = self.bottleneck.output_dim
            if backend == "frozen_quantum":
                for param in self.bottleneck.parameters():
                    param.requires_grad = False
        elif backend == "classical":
            bottleneck_out = CONFIG["num_qubits"] + len(CONFIG["readout_pairs"])
            self.bottleneck = nn.Sequential(
                nn.Linear(CONFIG["num_qubits"], CONFIG["head_hidden"]),
                nn.GELU(),
                nn.Linear(CONFIG["head_hidden"], bottleneck_out),
            )
        else:
            bottleneck_out = CONFIG["num_qubits"]
            self.bottleneck = nn.Identity()
        self.warmup_head = nn.Sequential(
            nn.Linear(CONFIG["num_qubits"] + CONFIG["skip_dim"], CONFIG["head_hidden"]),
            nn.GELU(),
            nn.Linear(CONFIG["head_hidden"], len(TARGET_COLS)),
        )
        self.head = nn.Sequential(
            nn.Linear(bottleneck_out + CONFIG["skip_dim"], CONFIG["head_hidden"]),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(CONFIG["head_hidden"], len(TARGET_COLS)),
        )

    def encode(self, aux: torch.Tensor, spectra: torch.Tensor, qfeat: torch.Tensor):
        aux_feat = self.aux_stem(aux)
        spec_feat = self.spec_stem(spectra)
        angles, skip, hidden = self.interface(aux_feat, spec_feat, qfeat)
        return angles, skip, hidden

    def forward(self, aux: torch.Tensor, spectra: torch.Tensor, qfeat: torch.Tensor, stage: str = "main"):
        angles, skip, hidden = self.encode(aux, spectra, qfeat)
        extras = {"angles": angles, "skip": skip, "hidden": hidden}
        if stage == "warmup":
            pred = self.warmup_head(torch.cat([angles, skip], dim=-1))
            extras["bottleneck"] = angles
            return pred, extras
        if self.backend == "quantum":
            bottleneck = self.bottleneck(angles)
        elif self.backend == "frozen_quantum":
            with torch.no_grad():
                bottleneck = self.bottleneck(angles)
            bottleneck = bottleneck.detach()
        elif self.backend == "classical":
            bottleneck = self.bottleneck(angles)
        else:
            bottleneck = self.bottleneck(angles)
        extras["bottleneck"] = bottleneck
        pred = self.head(torch.cat([bottleneck, skip], dim=-1))
        return pred, extras

    def classical_parameters(self):
        for module in (self.aux_stem, self.spec_stem, self.interface, self.warmup_head, self.head):
            yield from module.parameters()
        if self.backend == "classical":
            yield from self.bottleneck.parameters()

    def quantum_parameters(self):
        if self.backend == "quantum":
            yield from self.bottleneck.parameters()


class LegacyHybridReferenceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.aux_enc = AuxStem(len(AUX_FEATURE_COLS), 64, 32, CONFIG["dropout"])
        self.spec_enc = SpectralStem(64, 32, CONFIG["dropout"])
        self.fusion = nn.Sequential(
            nn.Linear(64, 48),
            nn.GELU(),
            nn.Linear(48, CONFIG["num_qubits"]),
            nn.LayerNorm(CONFIG["num_qubits"]),
        )
        self.quantum = ReuploadQuantumBlock(
            CONFIG["num_qubits"],
            num_blocks=1,
            readout_pairs=CONFIG["readout_pairs"],
            quantum_device_name=QUANTUM_DEVICE,
        )
        self.head = nn.Sequential(
            nn.Linear(self.quantum.output_dim + CONFIG["num_qubits"] + 64, 96),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(96, len(TARGET_COLS)),
        )
        self.residual = nn.Linear(self.quantum.output_dim, len(TARGET_COLS))

    def forward(self, aux: torch.Tensor, spectra: torch.Tensor, qfeat: torch.Tensor, stage: str = "main"):
        del qfeat
        aux_feat = self.aux_enc(aux)
        spec_feat = self.spec_enc(spectra)
        latent = torch.tanh(self.fusion(torch.cat([aux_feat, spec_feat], dim=-1))) * math.pi
        quantum_feat = self.quantum(latent)
        head_in = torch.cat([quantum_feat, latent, aux_feat, spec_feat], dim=-1)
        pred = self.head(head_in) + self.residual(quantum_feat)
        extras = {"angles": latent, "skip": torch.cat([aux_feat, spec_feat], dim=-1), "bottleneck": quantum_feat}
        return pred, extras

    def classical_parameters(self):
        for module in (self.aux_enc, self.spec_enc, self.fusion, self.head, self.residual):
            yield from module.parameters()

    def quantum_parameters(self):
        yield from self.quantum.parameters()


def count_parameters(model: nn.Module) -> dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}


for label in ["quantum", "classical", "frozen_quantum", "identity"]:
    model = BottleneckRegressor(backend=label)
    print(label, count_parameters(model))
print("legacy_hybrid", count_parameters(LegacyHybridReferenceModel()))

quantum {'total': 10486, 'trainable': 10486}
classical {'total': 11872, 'trainable': 11872}
frozen_quantum {'total': 10486, 'trainable': 10366}
identity {'total': 10078, 'trainable': 10078}
legacy_hybrid {'total': 37972, 'trainable': 37972}


## Training Utilities And Runtime Preflight

The hybrid path is still the expensive part, so the notebook includes:

- a one-batch forward/backward benchmark,
- staged warmup before full quantum training,
- separate classical and quantum learning rates,
- gradient and saturation diagnostics,
- identical evaluation code across all baselines.

In [5]:
def batch_iterator(split_tensors: dict[str, Any], batch_size: int, shuffle: bool, seed: int):
    n = len(split_tensors["indices"])
    order = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(order)
    for start in range(0, n, batch_size):
        batch_idx = order[start:start + batch_size]
        yield (
            split_tensors["aux"][batch_idx],
            split_tensors["spectra"][batch_idx],
            split_tensors["qfeat"][batch_idx],
            split_tensors["target"][batch_idx],
        )


def maybe_cap_split(split_tensors: dict[str, Any], limit: Optional[int], seed: int) -> dict[str, Any]:
    if limit is None or len(split_tensors["indices"]) <= limit:
        return split_tensors
    capped = subset_indices(split_tensors["indices"], limit=limit, seed=seed)
    local_positions = np.searchsorted(split_tensors["indices"], capped)
    return {
        key: value if key in {"indices", "planet_ids", "target_raw"} and not hasattr(value, "__getitem__") else value
        for key, value in split_tensors.items()
    }


def slice_split(split_tensors: dict[str, Any], positions: np.ndarray) -> dict[str, Any]:
    return {
        "indices": split_tensors["indices"][positions],
        "planet_ids": split_tensors["planet_ids"][positions],
        "aux": split_tensors["aux"][positions],
        "spectra": split_tensors["spectra"][positions],
        "qfeat": split_tensors["qfeat"][positions],
        "target": split_tensors["target"][positions],
        "target_raw": split_tensors["target_raw"][positions],
    }


def capped_split(split_tensors: dict[str, Any], limit: Optional[int], seed: int) -> dict[str, Any]:
    if limit is None or len(split_tensors["indices"]) <= limit:
        return split_tensors
    rng = np.random.default_rng(seed)
    positions = np.sort(rng.choice(len(split_tensors["indices"]), size=limit, replace=False))
    return slice_split(split_tensors, positions)


def build_low_data_bundle(base_bundle: dict[str, Any], train_size: int, seed: int) -> dict[str, Any]:
    rng = np.random.default_rng(seed)
    positions = np.sort(rng.choice(len(base_bundle["train"]["indices"]), size=train_size, replace=False))
    return {
        "name": f"{base_bundle['name']}_train_{train_size}",
        "train": slice_split(base_bundle["train"], positions),
        "val": base_bundle["val"],
        "test": base_bundle["test"],
    }


def grad_norm(parameters: Iterable[torch.nn.Parameter]) -> float:
    total = 0.0
    for param in parameters:
        if param.grad is None:
            continue
        total += float(param.grad.detach().pow(2).sum().item())
    return math.sqrt(total) if total > 0 else 0.0


def update_norm(parameters: Iterable[torch.nn.Parameter], snapshots: list[torch.Tensor]) -> float:
    total = 0.0
    for param, previous in zip(parameters, snapshots):
        delta = param.detach() - previous
        total += float(delta.pow(2).sum().item())
    return math.sqrt(total) if total > 0 else 0.0


def preflight_benchmark(model: nn.Module, split_tensors: dict[str, Any], batch_size: int, steps: int = 1) -> pd.DataFrame:
    model = model.to(DEVICE)
    rows = []
    criterion = nn.MSELoss()
    iterator = batch_iterator(split_tensors, batch_size=batch_size, shuffle=False, seed=SEED)
    for step_idx, batch in enumerate(iterator):
        if step_idx >= steps:
            break
        aux, spec, qfeat, tgt = [tensor.to(DEVICE) for tensor in batch]
        model.zero_grad(set_to_none=True)
        t0 = time.perf_counter()
        pred, extras = model(aux, spec, qfeat, stage="main")
        forward_s = time.perf_counter() - t0
        t1 = time.perf_counter()
        loss = criterion(pred, tgt)
        loss.backward()
        backward_s = time.perf_counter() - t1
        rows.append(
            {
                "step": step_idx + 1,
                "forward_s": forward_s,
                "backward_s": backward_s,
                "loss": float(loss.item()),
                "angle_std": float(extras["angles"].detach().std().item()),
                "angle_sat_frac": float((extras["angles"].detach().abs() > 0.9 * math.pi).float().mean().item()),
                "bottleneck_sat_frac": float((extras["bottleneck"].detach().abs() > 0.95).float().mean().item()),
            }
        )
    return pd.DataFrame(rows)


def evaluate_model(model: nn.Module, split_tensors: dict[str, Any], batch_size: int) -> tuple[float, np.ndarray, float, np.ndarray]:
    model.eval()
    criterion = nn.MSELoss()
    all_pred = []
    total_loss = 0.0
    total_count = 0
    with torch.inference_mode():
        for aux, spec, qfeat, tgt in batch_iterator(split_tensors, batch_size=batch_size, shuffle=False, seed=SEED):
            aux = aux.to(DEVICE)
            spec = spec.to(DEVICE)
            qfeat = qfeat.to(DEVICE)
            tgt = tgt.to(DEVICE)
            pred, _ = model(aux, spec, qfeat, stage="main")
            loss = criterion(pred, tgt)
            batch_count = len(aux)
            total_loss += float(loss.item()) * batch_count
            total_count += batch_count
            all_pred.append(pred.detach().cpu())
    pred_scaled = torch.cat(all_pred, dim=0).numpy()
    pred_orig = unscale_target(pred_scaled)
    rmse = np.sqrt(np.mean((pred_orig - split_tensors["target_raw"]) ** 2, axis=0))
    return total_loss / max(total_count, 1), rmse, float(rmse.mean()), pred_orig


def make_optimizer(model: nn.Module):
    classical_params = [p for p in model.classical_parameters() if p.requires_grad]
    quantum_params = [p for p in model.quantum_parameters() if p.requires_grad]
    groups = []
    if classical_params:
        groups.append({"params": classical_params, "lr": CONFIG["classical_lr"], "weight_decay": CONFIG["weight_decay"]})
    if quantum_params:
        groups.append({"params": quantum_params, "lr": 0.0, "weight_decay": 0.0})
    optimizer = torch.optim.AdamW(groups)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=CONFIG["scheduler_patience"],
    )
    return optimizer, scheduler, classical_params, quantum_params


def make_model(model_name: str) -> nn.Module:
    if model_name == "legacy_hybrid":
        return LegacyHybridReferenceModel()
    if model_name in {"quantum", "classical", "frozen_quantum", "identity"}:
        return BottleneckRegressor(backend=model_name)
    raise ValueError(f"Unknown model_name={model_name}")


def train_experiment(model_name: str, bundle: dict[str, Any], run_name: str) -> dict[str, Any]:
    model = make_model(model_name).to(DEVICE)
    optimizer, scheduler, classical_params, quantum_params = make_optimizer(model)
    criterion = nn.MSELoss()
    best_val = float("inf")
    best_state = None
    best_epoch = -1
    patience_left = CONFIG["early_stop"]
    history = []

    train_split = capped_split(bundle["train"], CONFIG["train_subset_cap"], SEED)
    val_split = capped_split(bundle["val"], CONFIG["val_subset_cap"], SEED)

    for epoch in range(CONFIG["max_epochs"]):
        model.train()
        train_losses = []
        epoch_stage = "main"
        quantum_lr = CONFIG["quantum_lr"]
        if model_name == "quantum":
            if epoch < CONFIG["warmup_epochs"]:
                epoch_stage = "warmup"
                quantum_lr = 0.0
            else:
                ramp_pos = min(epoch - CONFIG["warmup_epochs"] + 1, CONFIG["ramp_epochs"])
                quantum_lr = CONFIG["quantum_lr"] * min(1.0, ramp_pos / max(CONFIG["ramp_epochs"], 1))
        for group_idx, group in enumerate(optimizer.param_groups):
            if group_idx == 1:
                group["lr"] = quantum_lr

        quantum_snapshot = [param.detach().clone() for param in quantum_params]
        progress = tqdm(
            batch_iterator(train_split, batch_size=CONFIG["train_batch_size"], shuffle=True, seed=SEED + epoch),
            total=math.ceil(len(train_split["indices"]) / CONFIG["train_batch_size"]),
            desc=f"{run_name} | epoch {epoch + 1}/{CONFIG['max_epochs']}",
            leave=False,
        )
        for aux, spec, qfeat, tgt in progress:
            aux = aux.to(DEVICE)
            spec = spec.to(DEVICE)
            qfeat = qfeat.to(DEVICE)
            tgt = tgt.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred, extras = model(aux, spec, qfeat, stage=epoch_stage)
            loss = criterion(pred, tgt)
            loss.backward()
            if classical_params:
                torch.nn.utils.clip_grad_norm_(classical_params, CONFIG["gradient_clip"])
            if quantum_params:
                torch.nn.utils.clip_grad_norm_(quantum_params, CONFIG["quantum_gradient_clip"])
            optimizer.step()
            train_losses.append(float(loss.item()))
            progress.set_postfix(loss=f"{loss.item():.4f}", q_lr=f"{quantum_lr:.1e}")

        val_loss, val_rmse, val_rmse_mean, _ = evaluate_model(model, val_split, CONFIG["eval_batch_size"])
        scheduler.step(val_loss)
        row = {
            "epoch": epoch + 1,
            "stage": epoch_stage,
            "train_loss": float(np.mean(train_losses)) if train_losses else float("nan"),
            "val_loss": float(val_loss),
            "val_rmse_mean": float(val_rmse_mean),
            "lr_classical": optimizer.param_groups[0]["lr"],
            "lr_quantum": optimizer.param_groups[1]["lr"] if len(optimizer.param_groups) > 1 else 0.0,
            "classical_grad_norm": grad_norm(classical_params),
            "quantum_grad_norm": grad_norm(quantum_params),
            "quantum_update_norm": update_norm(quantum_params, quantum_snapshot) if quantum_params else 0.0,
            "angle_std": float(extras["angles"].detach().std().item()),
            "angle_sat_frac": float((extras["angles"].detach().abs() > 0.9 * math.pi).float().mean().item()),
            "bottleneck_sat_frac": float((extras["bottleneck"].detach().abs() > 0.95).float().mean().item()),
        }
        for target_name, target_rmse in zip(TARGET_COLS, val_rmse):
            row[f"rmse_{target_name}"] = float(target_rmse)
        history.append(row)

        if val_loss < best_val:
            best_val = val_loss
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch + 1
            patience_left = CONFIG["early_stop"]
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    test_loss, test_rmse, test_rmse_mean, test_pred = evaluate_model(model, bundle["test"], CONFIG["eval_batch_size"])
    val_loss, val_rmse, val_rmse_mean, val_pred = evaluate_model(model, bundle["val"], CONFIG["eval_batch_size"])
    metrics = pd.DataFrame(
        [
            {"model": model_name, "bundle": bundle["name"], "split": "val", "mean_rmse": val_rmse_mean, **{f"rmse_{c}": float(r) for c, r in zip(TARGET_COLS, val_rmse)}},
            {"model": model_name, "bundle": bundle["name"], "split": "test", "mean_rmse": test_rmse_mean, **{f"rmse_{c}": float(r) for c, r in zip(TARGET_COLS, test_rmse)}},
        ]
    )
    prediction_frame = pd.DataFrame({"planet_ID": bundle["test"]["planet_ids"]})
    for i, target_name in enumerate(TARGET_COLS):
        prediction_frame[f"true_{target_name}"] = bundle["test"]["target_raw"][:, i]
        prediction_frame[f"pred_{target_name}"] = test_pred[:, i]
    return {
        "model": model,
        "history": pd.DataFrame(history),
        "metrics": metrics,
        "test_predictions": prediction_frame,
        "best_epoch": best_epoch,
        "best_val_loss": best_val,
        "val_predictions": val_pred,
        "test_predictions_array": test_pred,
    }


benchmark_model = make_model("quantum")
preflight_df = preflight_benchmark(
    benchmark_model,
    capped_split(iid_bundle["train"], CONFIG["train_batch_size"], SEED),
    batch_size=CONFIG["train_batch_size"],
    steps=CONFIG["preflight_batches"],
)
display(preflight_df)

,step,forward_s,backward_s,loss,angle_std,angle_sat_frac,bottleneck_sat_frac
0,1,3.378999,3.495291,0.937717,2.094022,0.114583,0.082176


## Experiment Matrix

The default suite compares:

- `quantum`: the new 12-qubit bottleneck model,
- `classical`: a matched classical bottleneck,
- `frozen_quantum`: same quantum feature map with frozen weights,
- `identity`: no learned bottleneck transform,
- `legacy_hybrid`: a reference model close to the original notebook.

It evaluates IID full-data performance, low-data subsets, and an OOD split based on the upper quantile of `log_H2O`.

In [6]:
EXPERIMENT_BUNDLES = {"iid_full": iid_bundle, "ood_high_h2o": ood_bundle}
for train_size in CONFIG["low_data_sizes"]:
    if train_size < len(iid_bundle["train"]["indices"]):
        EXPERIMENT_BUNDLES[f"iid_train_{train_size}"] = build_low_data_bundle(iid_bundle, train_size, SEED + train_size)

MODEL_ORDER = ["quantum", "classical", "frozen_quantum", "identity", "legacy_hybrid"]

print("Available bundles:")
for bundle_name, bundle in EXPERIMENT_BUNDLES.items():
    print(
        f" - {bundle_name}: train={len(bundle['train']['indices']):,}, "
        f"val={len(bundle['val']['indices']):,}, test={len(bundle['test']['indices']):,}"
    )
print("Models:", MODEL_ORDER)


def run_experiment_suite(model_order: list[str], experiment_bundles: dict[str, dict[str, Any]]):
    suite_rows = []
    histories = {}
    predictions = {}
    checkpoints = {}
    for bundle_name, bundle in experiment_bundles.items():
        for model_name in model_order:
            run_name = f"{bundle_name}:{model_name}"
            print(f"Running {run_name}")
            result = train_experiment(model_name, bundle, run_name)
            suite_rows.append(result["metrics"])
            histories[run_name] = result["history"]
            predictions[run_name] = result["test_predictions"]
            checkpoints[run_name] = {
                "best_epoch": result["best_epoch"],
                "best_val_loss": result["best_val_loss"],
                "state_dict": deepcopy(result["model"].state_dict()),
            }
    metrics = pd.concat(suite_rows, ignore_index=True)
    return {
        "metrics": metrics,
        "histories": histories,
        "predictions": predictions,
        "checkpoints": checkpoints,
    }


suite_artifacts = None
if CONFIG["run_experiments"]:
    suite_artifacts = run_experiment_suite(MODEL_ORDER, EXPERIMENT_BUNDLES)
    display(suite_artifacts["metrics"].sort_values(["bundle", "split", "mean_rmse"]))
else:
    print("Set CONFIG['run_experiments'] = True and rerun this cell to execute the full suite.")

Available bundles:
 - iid_full: train=33,552, val=3,728, test=4,143
 - ood_high_h2o: train=31,688, val=3,521, test=6,214
 - iid_train_500: train=500, val=3,728, test=4,143
 - iid_train_1000: train=1,000, val=3,728, test=4,143
 - iid_train_2000: train=2,000, val=3,728, test=4,143
 - iid_train_5000: train=5,000, val=3,728, test=4,143
Models: ['quantum', 'classical', 'frozen_quantum', 'identity', 'legacy_hybrid']
Set CONFIG['run_experiments'] = True and rerun this cell to execute the full suite.


In [7]:
if suite_artifacts is not None:
    metrics_df = suite_artifacts["metrics"].copy()
    display(metrics_df)

    test_metrics = metrics_df[metrics_df["split"] == "test"].copy()
    pivot = test_metrics.pivot(index="bundle", columns="model", values="mean_rmse")
    display(pivot)

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel("Mean RMSE")
    ax.set_title("Test Mean RMSE Across Bundles")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    if CONFIG["save_artifacts"]:
        plt.savefig(CONFIG["output_dir"] / "bundle_comparison_rmse.png", dpi=150)
    plt.show()

    low_data_df = test_metrics[test_metrics["bundle"].str.startswith("iid_train_")].copy()
    if not low_data_df.empty:
        low_data_df["train_size"] = low_data_df["bundle"].str.replace("iid_train_", "", regex=False).astype(int)
        fig, ax = plt.subplots(figsize=(10, 5))
        for model_name in MODEL_ORDER:
            model_slice = low_data_df[low_data_df["model"] == model_name].sort_values("train_size")
            if not model_slice.empty:
                ax.plot(model_slice["train_size"], model_slice["mean_rmse"], marker="o", label=model_name)
        ax.set_xscale("log")
        ax.set_xlabel("Training subset size")
        ax.set_ylabel("Test mean RMSE")
        ax.set_title("Low-data scaling")
        ax.legend()
        plt.tight_layout()
        if CONFIG["save_artifacts"]:
            plt.savefig(CONFIG["output_dir"] / "low_data_scaling.png", dpi=150)
        plt.show()

    history_frames = []
    for run_name, history_df in suite_artifacts["histories"].items():
        temp = history_df.copy()
        temp["run_name"] = run_name
        history_frames.append(temp)
    all_history = pd.concat(history_frames, ignore_index=True)
    display(all_history.head())
else:
    print("No suite artifacts yet. Run the experiment suite first.")

No suite artifacts yet. Run the experiment suite first.


In [8]:
if suite_artifacts is not None and CONFIG["save_artifacts"]:
    metrics_path = CONFIG["output_dir"] / "suite_metrics.csv"
    metrics_df = suite_artifacts["metrics"].copy()
    metrics_df.to_csv(metrics_path, index=False)

    history_dir = CONFIG["output_dir"] / "histories"
    prediction_dir = CONFIG["output_dir"] / "predictions"
    checkpoint_dir = CONFIG["output_dir"] / "checkpoints"
    history_dir.mkdir(parents=True, exist_ok=True)
    prediction_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    for run_name, history_df in suite_artifacts["histories"].items():
        history_df.to_csv(history_dir / f"{run_name.replace(':', '__')}.csv", index=False)
    for run_name, prediction_df in suite_artifacts["predictions"].items():
        prediction_df.to_csv(prediction_dir / f"{run_name.replace(':', '__')}.csv", index=False)
    for run_name, checkpoint in suite_artifacts["checkpoints"].items():
        torch.save(checkpoint, checkpoint_dir / f"{run_name.replace(':', '__')}.pt")

    metadata = {
        "config": {key: str(value) if isinstance(value, Path) else value for key, value in CONFIG.items()},
        "feature_names": QUANTUM_FEATURE_NAMES,
        "target_cols": TARGET_COLS,
        "aux_feature_cols": AUX_FEATURE_COLS,
        "ood_threshold": ood_threshold,
    }
    (CONFIG["output_dir"] / "run_metadata.json").write_text(json.dumps(metadata, indent=2))

    print(f"Saved suite artifacts to {CONFIG['output_dir']}")
else:
    print("Artifact save skipped. Either the suite has not run yet or CONFIG['save_artifacts'] is False.")

Artifact save skipped. Either the suite has not run yet or CONFIG['save_artifacts'] is False.
